<div align="center">

# Classificador de Imagens Astronômicas usando CNN

| Disciplina | Semestre | Docente | Horário |
| :---: | :---: | :---: | :---: |
| PAM0466 - SISTEMAS INTELIGENTES | 2026.1 | PEDRO THIAGO VALÉRIO DE SOUZA | 3M23 4M45 |

| Discente | Matrícula |
| :---: | :---: |
| ELTON CAIO VIEIRA DE LIMA | 2020010673 |
| LUCAS VIERES ARAÚJO FARIAS | 2025022531 |
</div>

Projeto da disciplina de Sistemas Inteligentes, em que foi utilizado um modelo de Redes Neurais Convolucionais (CNN) para a classificação de imagens astronômicas.

<hr>

Dataset utilizado: Astrophysical_Objects_Image_Dataset_Maxia_E, by: Edoardo Maxia.
</p>
Devido ao tamanho do dataset, não foi possivel anexá-lo a este git, portanto, segue o link para download do mesmo:
</p>

https://www.kaggle.com/datasets/engeddy/astrophysical-objects-image-dataset

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, recall_score
import matplotlib.pyplot as plt


Fazendo o transform para padrozinar todas as entradas do dataset.

In [3]:
train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])


Carregando o dataset já pré setado em treino, validação e teste.

In [5]:
train_dataset = ImageFolder(
    root="astro_dataset_maxia/training",
    transform=train_transform
)

val_dataset = ImageFolder(
    root="astro_dataset_maxia/validation",
    transform=test_transform
)

test_dataset = ImageFolder(
    root="astro_dataset_maxia/test",
    transform=test_transform
)


print(train_dataset.classes)

print("Treino:", len(train_dataset))
print("Validação:", len(val_dataset))
print("Teste:", len(test_dataset))

['asteroid', 'black_hole', 'earth', 'galaxy', 'jupiter', 'mars', 'mercury', 'neptune', 'pluto', 'saturn', 'uranus', 'venus']
Treino: 2416
Validação: 658
Teste: 345


Definindo os batches do treino, validação e teste. </p>
Proporção de amostras de cada etapa: </p>
Treino: 70% </p>
Validação: 20% </p>
Teste: 10% 

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print(f"Treino:    {len(train_dataset)}")
print(f"Validação: {len(val_dataset)}")
print(f"Teste:     {len(test_dataset)}")

print(f"Batches treino: {len(train_loader)}")
print(f"Batches validação: {len(val_loader)}")
print(f"Batches teste: {len(test_loader)}")

Treino:    2416
Validação: 658
Teste:     345
Batches treino: 151
Batches validação: 42
Batches teste: 22


Definindo as camadas convolucionais do modelo.

In [7]:
class AstroCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

Configurando métricas do treinamento do modelo.

In [8]:
model = AstroCNN(
    num_classes=len(train_dataset.classes)
)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

epochs = 50
patience = 5

best_val_loss = float("inf")
best_weights = None
patience_count = 0

train_losses = []
val_losses = []

train_accs = []
val_accs = []

Treinamento do modelo

In [9]:
for epoch in range(1, epochs + 1):
    
    model.train()
    train_loss, train_correct = 0.0, 0
    for X, y in train_loader:
        X, y = X, y
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * len(X)
        train_correct += (logits.argmax(dim=1) == y).sum().item()
    
    train_loss  = train_loss/len(train_dataset)
    train_acc   = train_correct/len(train_dataset)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X, y
            logits      = model(X)
            val_loss   += criterion(logits, y).item() * len(X)
            val_correct += (logits.argmax(dim=1) == y).sum().item()
    

    val_loss  = val_loss/len(val_dataset)
    val_acc   = val_correct / len(val_dataset)  
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch:2d}/{epochs} "
          f"| train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} "
          f"| val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")


    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_weights   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_count = 0
        print(f" ✔ val_loss melhorou → {best_val_loss:.6f}")
    else:
        patience_count += 1
        print(f" ✘ sem melhora há {patience_count}/{patience} épocas")
        if patience_count >= patience:
            print(f"\nEarly stopping acionado na época {epoch}.")
            break


model.load_state_dict(best_weights)
print("Melhores pesos restaurados.")

Epoch  1/50 | train_loss: 1.5432 train_acc: 0.4719 | val_loss: 0.9470 val_acc: 0.6748
 ✔ val_loss melhorou → 0.946960
Epoch  2/50 | train_loss: 0.7545 train_acc: 0.7550 | val_loss: 0.6253 val_acc: 0.8009
 ✔ val_loss melhorou → 0.625255
Epoch  3/50 | train_loss: 0.5442 train_acc: 0.8204 | val_loss: 0.4609 val_acc: 0.8602
 ✔ val_loss melhorou → 0.460870
Epoch  4/50 | train_loss: 0.3929 train_acc: 0.8684 | val_loss: 0.4218 val_acc: 0.8891
 ✔ val_loss melhorou → 0.421777
Epoch  5/50 | train_loss: 0.3264 train_acc: 0.8924 | val_loss: 0.3343 val_acc: 0.8997
 ✔ val_loss melhorou → 0.334272
Epoch  6/50 | train_loss: 0.2340 train_acc: 0.9172 | val_loss: 0.3344 val_acc: 0.8967
 ✘ sem melhora há 1/5 épocas
Epoch  7/50 | train_loss: 0.1964 train_acc: 0.9334 | val_loss: 0.3735 val_acc: 0.8906
 ✘ sem melhora há 2/5 épocas
Epoch  8/50 | train_loss: 0.1532 train_acc: 0.9499 | val_loss: 0.3756 val_acc: 0.9271
 ✘ sem melhora há 3/5 épocas
Epoch  9/50 | train_loss: 0.1337 train_acc: 0.9578 | val_loss: 0.

Avaliação das métricas de precisão do modelo.

In [10]:
y_true = []
y_pred = []

model.eval()

test_loss = 0.0
test_correct = 0

with torch.no_grad():
    for X, y in test_loader:

        logits = model(X)
        preds = logits.argmax(dim=1)

        y_true.extend(y.numpy())
        y_pred.extend(preds.numpy())

        test_loss += criterion(logits, y).item() * len(X)

        test_correct += (preds == y).sum().item()

test_loss = test_loss / len(test_dataset)
f1 = f1_score(y_true, y_pred, average = 'macro')
recall = recall_score(y_true, y_pred, average = 'macro')
accuracy = accuracy_score(y_true, y_pred)

print(f"\nLoss: {test_loss:.4f} | Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f} |  F1-score: {f1:.4f}")

print(f"\n        ──Matriz de confusão──")
cmatrix = confusion_matrix(y_true, y_pred)
print (cmatrix)


Loss: 0.3045 | Accuracy: 0.9159
Recall: 0.9164 |  F1-score: 0.9185

        ──Matriz de confusão──
[[25  3  0  1  0  0  0  0  0  0  0  0]
 [ 2 28  0  1  0  0  0  0  0  0  0  0]
 [ 1  1 25  1  0  0  0  1  0  0  0  0]
 [ 1  2  0 27  0  0  0  0  0  0  0  0]
 [ 0  0  0  0 26  0  0  0  2  0  0  0]
 [ 0  2  0  0  0 28  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 27  0  1  0  0  0]
 [ 0  0  1  0  0  0  0 27  0  0  0  0]
 [ 1  0  0  0  1  0  0  0 26  0  0  0]
 [ 2  0  0  0  0  0  0  0  0 26  0  0]
 [ 0  0  0  0  0  0  0  1  0  0 27  0]
 [ 0  0  0  0  0  4  0  0  0  0  0 24]]
